##FEATURE ENGINEERING

##ORDER AND SHIPDATE

In [ ]:
##IMPORTING LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

In [ ]:
##LOADING DATASET

df_clean = pd.read_csv("../data/raw/superstore.csv", encoding="cp1252")

print("Data loaded successfully!")


TIME

In [46]:
df_clean["Order Date"] = pd.to_datetime(df_clean["Order Date"])
df_clean["Ship Date"] = pd.to_datetime(df_clean["Ship Date"])

In [47]:
df_clean["Year"] = df_clean["Order Date"].dt.year
df_clean["Month"] = df_clean["Order Date"].dt.month
df_clean["Month_Name"] = df_clean["Order Date"].dt.month_name()
df_clean["Quarter"] = df_clean["Order Date"].dt.quarter
df_clean["Week"] = df_clean["Order Date"].dt.isocalendar().week
df_clean["Day_of_Week"] = df_clean["Order Date"].dt.dayofweek
df_clean["Day_Name"] = df_clean["Order Date"].dt.day_name()


In [48]:
#3CREATING A MONTH-YEAR FEATURE

df_clean["Month_Year"] = df_clean["Order Date"].dt.to_period("M")

In [49]:
##CREATING WEEKEND INFORMATION
df_clean["Is_Weekend"] = (df_clean["Order Date"].dt.dayofweek >=5).astype(int)

OPERATIONAL

In [50]:
##CREATING SHIPPING DURATION
df_clean["Shipping_Day"] = (df_clean["Ship Date"] - df_clean["Order Date"]).dt.days

BUSINESS

In [52]:
##CALCULATING PROFIT MARGIN
df_clean["Profit_Margin"] = np.where(
    df_clean["Sales"] != 0,
    df_clean["Profit"] / df_clean["Sales"],
    0
)

In [53]:
##CALCULATION SALES PER UNIT
df_clean["Sales_Per_Unit"] = np.where(
    df_clean["Quantity"] != 0,
    df_clean["Sales"] / df_clean["Quantity"],
    0
)

In [54]:
##PROFIT PER UNIT
df_clean["Profit_Per_Unit"] = np.where(
    df_clean["Quantity"] != 0,
    df_clean["Profit"] / df_clean["Quantity"],
    0
)

In [56]:
##EVALUATING DISCOUNT LEVEL 
df_clean["Discount_Level"] = pd.cut(
    df_clean["Discount"],
    bins=[-0.01, 0, 0.10, 0.20, 0.50, 1.00],
    labels=[
        "No Discount",
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

we already have
Category
Sub-Category

These are important because different product groups can have completely different sales behaviour.

There is no need to create new features from them immediately.

But later, when preparing the ML dataset, we'll need to encode them.

For example
Category
Furniture
Technology
Office Supplies

can be transformed using:

One-hot encoding
Target encoding, depending on the model/design

We'll deal with that during the ML preprocessing stage.

we also have

segment that tells us about what type of customer made the purchase
eg
Consumer
Corporate
Home Office

Again, we'll test its predictive value rather than assuming it.

HISTORICAL FORECASTING

In [57]:
##CREATING A MONTHLY FORECASTING DATASET
monthly_sales = (
    df_clean
    .groupby("Month_Year")
    .agg({
        "Sales": "sum",
        "Quantity": "sum",
        "Profit": "sum",
        "Discount": "mean"
    })
    .reset_index()
)

In [58]:
monthly_sales["Year"] = (monthly_sales["Month_Year"].dt.year)
monthly_sales["Month"] = (monthly_sales["Month_Year"].dt.month)

In [59]:
##CREATING LAG FEATURES
#Sales from the previous month

monthly_sales["Sales_Lag_1"] = monthly_sales["Sales"].shift(1)

In [61]:
#Sales from the previous 3 months
monthly_sales["Sales_Lag_3"] = monthly_sales["Sales"].shift(3)

In [62]:
#Sales from the previous 6 months
monthly_sales["Sales_Lag_6"] = monthly_sales["Sales"].shift(6)

In [63]:
#Sales from the previous 12 months

monthly_sales["Sales_Lag_12"] = monthly_sales["Sales"].shift(12)

The 12-month lag is especially interesting because it allows the model to compare a month with the same month in the previous year.

In [64]:
##CREATING ROLLING AVERAGES
#Rolling average of the previous 3 months
monthly_sales["Sales_Rolling_3"] = monthly_sales["Sales"].shift(1).rolling(window=3).mean()

#Rolling average of the previous 6 months
monthly_sales["Sales_Rolling_6"] = monthly_sales["Sales"].shift(1).rolling(window=6).mean()

#Rolling average of the previous 12 months
monthly_sales["Sales_Rolling_12"] = monthly_sales["Sales"].shift(1).rolling(window=12).mean()

These capture the recent sales trend.

In [ ]:
df_clean.head()